# Hyperliquid Trader Performance × Bitcoin Fear & Greed Index
### Data Science Assignment — Web3 Trading Analysis

**Datasets used:**
- `fear_greed_index.csv` — Bitcoin Fear & Greed Index (2018–2025): `timestamp`, `value`, `classification`, `date`
- `historical_data.csv` — Hyperliquid perpetuals data (2023–2025): 211,224 rows × 16 columns

**Key columns in historical data:** `Account`, `Coin`, `Execution Price`, `Size Tokens`, `Size USD`, `Side`, `Timestamp IST`, `Start Position`, `Direction`, `Closed PnL`, `Fee`, `Trade ID`, `Timestamp`


In [ ]:
# ── 0. Imports ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu, binomtest
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#fafafa',
    'axes.grid': True, 'grid.alpha': 0.25,
    'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False,
})

SENT_ORDER  = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
SENT_COLORS = {'Extreme Fear':'#D85A30','Fear':'#F09595','Neutral':'#B4B2A9',
               'Greed':'#97C459','Extreme Greed':'#1D9E75'}
print("✓ Setup complete")


## 1. Data Loading & Preprocessing

In [ ]:
# ── 1. Load real CSVs ────────────────────────────────────────────────────────
hist = pd.read_csv('historical_data.csv')
fg   = pd.read_csv('fear_greed_index.csv')

# Parse IST timestamp (dd-mm-yyyy HH:MM format) for accurate daily dates
hist['datetime'] = pd.to_datetime(hist['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce')
hist['date']     = hist['datetime'].dt.date.astype(str)
fg['date']       = fg['date'].astype(str)

# Categorical sentiment for ordered groupbys
fg['classification'] = pd.Categorical(fg['classification'], categories=SENT_ORDER, ordered=True)

print(f"Historical data : {hist.shape[0]:,} rows × {hist.shape[1]} columns")
print(f"Fear/Greed index: {fg.shape[0]:,} rows | {fg['date'].min()} → {fg['date'].max()}")
print(f"Trade date range: {hist['date'].min()} → {hist['date'].max()}")
print(f"Unique accounts : {hist['Account'].nunique()}")
print(f"Unique coins    : {hist['Coin'].nunique()}")
hist.head(3)


In [ ]:
# ── 1b. Direction & PnL overview ─────────────────────────────────────────────
print("Direction breakdown:")
print(hist['Direction'].value_counts().to_string())
print(f"\nClosed PnL stats (all rows):")
print(hist['Closed PnL'].describe().round(2).to_string())
print(f"\nNon-zero PnL rows (closing trades): {(hist['Closed PnL'] != 0).sum():,}")


In [ ]:
# ── 1c. Build the analysis dataset ───────────────────────────────────────────
# Keep only rows where a position was closed (non-zero PnL)
close_trades = hist[hist['Closed PnL'] != 0].copy()

# Merge with Fear/Greed on date
merged = close_trades.merge(fg[['date','value','classification']], on='date', how='left')
merged = merged.dropna(subset=['classification'])
merged['classification'] = pd.Categorical(merged['classification'], categories=SENT_ORDER, ordered=True)
merged['win'] = (merged['Closed PnL'] > 0).astype(int)
merged['net_pnl'] = merged['Closed PnL'] - merged['Fee']

print(f"Closing trades after merge: {len(merged):,}")
print(f"Sentiment coverage: {merged['classification'].notna().mean():.1%}")
print(f"\nTrade count per sentiment:")
print(merged['classification'].value_counts().reindex(SENT_ORDER))


## 2. Exploratory Data Analysis

In [ ]:
# ── 2a. Fear/Greed distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(fg['value'], bins=30, color='#378ADD', alpha=0.85, edgecolor='white')
for x, lbl in [(25,'Extreme Fear'),(45,'Fear'),(55,'Neutral'),(75,'Greed')]:
    axes[0].axvline(x, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
axes[0].set_title('Fear & Greed Index distribution (2018–2025)', fontweight='bold')
axes[0].set_xlabel('FG value'); axes[0].set_ylabel('Days')

counts = fg['classification'].value_counts().reindex(SENT_ORDER)
colors_list = [SENT_COLORS[c] for c in SENT_ORDER]
axes[1].barh(SENT_ORDER, counts.values, color=colors_list)
axes[1].set_title('Days per sentiment regime', fontweight='bold')
for i, v in enumerate(counts.values):
    axes[1].text(v+3, i, str(v), va='center', fontsize=10)

plt.tight_layout(); plt.savefig('eda_fg_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2b. Trader data overview ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Top coins by trade count
top10 = hist['Coin'].value_counts().head(10)
axes[0].barh(top10.index[::-1], top10.values[::-1], color='#7F77DD')
axes[0].set_title('Top 10 coins by trade count', fontweight='bold')
axes[0].set_xlabel('Trades')

# PnL distribution (clip outliers for visibility)
closed_pnl = hist[hist['Closed PnL'] != 0]['Closed PnL']
axes[1].hist(closed_pnl.clip(-3000, 3000), bins=80, color='#1D9E75', alpha=0.85, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.2)
axes[1].set_title('Closed PnL distribution (clipped ±$3K)', fontweight='bold')
axes[1].set_xlabel('PnL (USD)')

# Side distribution
side_counts = hist['Side'].value_counts()
axes[2].pie(side_counts, labels=side_counts.index, autopct='%1.1f%%',
            colors=['#378ADD','#D85A30'], startangle=90)
axes[2].set_title('BUY vs SELL split', fontweight='bold')

plt.tight_layout(); plt.savefig('eda_trader_overview.png', dpi=150, bbox_inches='tight')
plt.show()

profitable = (closed_pnl > 0).mean()
print(f"Overall win rate on closed trades: {profitable:.1%}")
print(f"Median PnL: ${closed_pnl.median():.2f} | Mean: ${closed_pnl.mean():.2f}")


## 3. Core Analysis — Trader PnL vs Market Sentiment

In [ ]:
# ── 3a. PnL & Win Rate by Sentiment ─────────────────────────────────────────
summary = merged.groupby('classification', observed=False).agg(
    avg_pnl      = ('Closed PnL', 'mean'),
    median_pnl   = ('Closed PnL', 'median'),
    total_pnl    = ('Closed PnL', 'sum'),
    win_rate     = ('win',        'mean'),
    trade_count  = ('Closed PnL', 'count'),
    std_pnl      = ('Closed PnL', 'std'),
    avg_net_pnl  = ('net_pnl',    'mean'),
).round(2)
summary['win_rate_pct'] = (summary['win_rate'] * 100).round(1)
print("=== Core sentiment summary ===")
print(summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_list = [SENT_COLORS[c] for c in SENT_ORDER]

bars = axes[0].bar(SENT_ORDER, summary['avg_pnl'], color=colors_list, edgecolor='white')
axes[0].set_title('Avg closed PnL per trade by sentiment', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Avg PnL (USD)')
axes[0].set_xticklabels(SENT_ORDER, rotation=12, ha='right')
for b, v in zip(bars, summary['avg_pnl']):
    axes[0].text(b.get_x()+b.get_width()/2, v+1, f'${v:.0f}', ha='center', fontsize=10, fontweight='500')

bars2 = axes[1].bar(SENT_ORDER, summary['win_rate_pct'], color=colors_list, edgecolor='white')
axes[1].axhline(50, color='red', linewidth=1, linestyle='--', alpha=0.5, label='50% baseline')
axes[1].set_ylim(60, 95); axes[1].set_title('Win rate % by sentiment', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Win Rate %'); axes[1].set_xticklabels(SENT_ORDER, rotation=12, ha='right')
axes[1].legend(fontsize=9)
for b, v in zip(bars2, summary['win_rate_pct']):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.2, f'{v}%', ha='center', fontsize=10, fontweight='500')

plt.tight_layout(); plt.savefig('pnl_by_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3b. Trade Size by Sentiment ──────────────────────────────────────────────
size_sent = merged.groupby('classification', observed=False)['Size USD'].agg(['mean','median']).round(2)
print("Avg trade size USD by sentiment:")
print(size_sent)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(SENT_ORDER, size_sent['mean'], color=colors_list, edgecolor='white')
ax.set_title('Avg trade size (USD) by sentiment — traders bet bigger in Fear', fontweight='bold')
ax.set_ylabel('Avg Size USD'); ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax.set_xticklabels(SENT_ORDER, rotation=12, ha='right')
for i, v in enumerate(size_sent['mean']):
    ax.text(i, v+50, f'${v:,.0f}', ha='center', fontsize=10)
plt.tight_layout(); plt.savefig('trade_size_by_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. BUY vs SELL Analysis by Sentiment

In [ ]:
# ── 4. Side × Sentiment ──────────────────────────────────────────────────────
side_pnl = merged.groupby(['classification','Side'], observed=False)['Closed PnL'].mean().unstack().round(2)
side_wr  = merged.groupby(['classification','Side'], observed=False)['win'].mean().unstack().round(3) * 100
print("Avg PnL — BUY vs SELL per sentiment:")
print(side_pnl)
print("\nWin Rate % — BUY vs SELL per sentiment:")
print(side_wr.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(SENT_ORDER)); w = 0.35
buy_pnl  = [side_pnl.loc[s,'BUY']  if s in side_pnl.index else 0 for s in SENT_ORDER]
sell_pnl = [side_pnl.loc[s,'SELL'] if s in side_pnl.index else 0 for s in SENT_ORDER]
axes[0].bar(x-w/2, buy_pnl,  w, label='BUY',  color='#378ADD', edgecolor='white')
axes[0].bar(x+w/2, sell_pnl, w, label='SELL', color='#D85A30', edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.7, linestyle='--')
axes[0].set_xticks(x); axes[0].set_xticklabels(SENT_ORDER, rotation=12, ha='right')
axes[0].set_title('Avg PnL: BUY vs SELL by sentiment', fontweight='bold'); axes[0].legend()
axes[0].set_ylabel('Avg PnL (USD)')

dir_pnl = merged[merged['Direction'].isin(['Close Long','Close Short','Sell'])].groupby('Direction').agg(
    avg_pnl=('Closed PnL','mean'), win_rate=('win','mean'), count=('Closed PnL','count')).round(2)
axes[1].bar(dir_pnl.index, dir_pnl['avg_pnl'], color=['#378ADD','#7F77DD','#1D9E75'], edgecolor='white')
axes[1].set_title('Avg PnL by trade direction type', fontweight='bold'); axes[1].set_ylabel('Avg PnL (USD)')
for i, (idx, row) in enumerate(dir_pnl.iterrows()):
    axes[1].text(i, row['avg_pnl']+1, f"${row['avg_pnl']:.0f}\n(wr {row['win_rate']*100:.0f}%)", ha='center', fontsize=9)

plt.tight_layout(); plt.savefig('side_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nKey: BUYs win in Fear ($210 avg), SELLs win in Extreme Greed ($176 avg)")


## 5. Temporal Analysis — Monthly Trend

In [ ]:
# ── 5. Monthly trend ─────────────────────────────────────────────────────────
merged['month'] = merged['datetime'].dt.to_period('M')
fg['month']     = pd.to_datetime(fg['date']).dt.to_period('M')
monthly_fg  = fg.groupby('month')['value'].mean().reset_index()
monthly_pnl = merged.groupby('month').agg(
    avg_pnl=('Closed PnL','mean'), count=('Closed PnL','count')).reset_index()
monthly = monthly_fg.merge(monthly_pnl, on='month').query("count > 20").copy()
monthly['month_str'] = monthly['month'].astype(str)

# Pearson correlation
r, p = stats.pearsonr(monthly['value'], monthly['avg_pnl'])
print(f"Pearson r(FG index, avg PnL) = {r:.3f}  p = {p:.3f}")

fig, ax1 = plt.subplots(figsize=(15, 5))
ax2 = ax1.twinx()
ax1.fill_between(monthly['month_str'], monthly['value'], alpha=0.1, color='#378ADD')
ax1.plot(monthly['month_str'], monthly['value'], color='#378ADD', linewidth=2, marker='o', markersize=4, label='Fear/Greed index')
ax2.plot(monthly['month_str'], monthly['avg_pnl'], color='#1D9E75', linewidth=2, linestyle='--', marker='s', markersize=4, label='Avg PnL')
ax2.axhline(0, color='gray', linewidth=0.7, linestyle=':')
ax1.set_ylabel('Fear/Greed index', color='#378ADD'); ax1.set_ylim(0, 100)
ax2.set_ylabel('Avg monthly PnL (USD)', color='#1D9E75')
ax1.set_xticklabels(monthly['month_str'], rotation=45, ha='right', fontsize=8)
ax1.set_title(f'Monthly FG Index vs Avg Trader PnL | Pearson r={r:.3f}', fontweight='bold', fontsize=13)
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lbl1+lbl2, loc='upper left', fontsize=9)
plt.tight_layout(); plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Coin-Level Analysis

In [ ]:
# ── 6. Coin performance ──────────────────────────────────────────────────────
coin_summary = merged.groupby('Coin').agg(
    avg_pnl    = ('Closed PnL', 'mean'),
    total_pnl  = ('Closed PnL', 'sum'),
    win_rate   = ('win',        'mean'),
    count      = ('Closed PnL', 'count'),
    avg_size   = ('Size USD',   'mean'),
).round(2).sort_values('count', ascending=False)
coin_summary['win_rate_pct'] = (coin_summary['win_rate']*100).round(1)
print("=== Top 15 coins ===")
print(coin_summary.head(15).to_string())

top8 = coin_summary.head(8)
coin_colors = ['#378ADD','#7F77DD','#F09595','#1D9E75','#97C459','#D4537E','#D85A30','#B4B2A9']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(top8.index, top8['count'], color=coin_colors, edgecolor='white')
axes[0].set_title('Trade count — top 8 coins', fontweight='bold')
axes[0].set_ylabel('Trades'); axes[0].set_xticklabels(top8.index, rotation=20, ha='right')

bars = axes[1].bar(top8.index, top8['avg_pnl'], color=coin_colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.7, linestyle='--')
axes[1].set_title('Avg PnL per trade — top 8 coins', fontweight='bold')
axes[1].set_ylabel('Avg PnL (USD)'); axes[1].set_xticklabels(top8.index, rotation=20, ha='right')
for b, v in zip(bars, top8['avg_pnl']):
    axes[1].text(b.get_x()+b.get_width()/2, v+(5 if v>=0 else -12), f'${v:.0f}', ha='center', fontsize=9)

plt.tight_layout(); plt.savefig('coin_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Account-Level Profitability

In [ ]:
# ── 7. Account rankings ──────────────────────────────────────────────────────
acc_summary = merged.groupby('Account').agg(
    total_pnl   = ('Closed PnL', 'sum'),
    avg_pnl     = ('Closed PnL', 'mean'),
    win_rate    = ('win',        'mean'),
    trade_count = ('Closed PnL', 'count'),
    coins_traded= ('Coin',       'nunique'),
).round(2).sort_values('total_pnl', ascending=False)
acc_summary['win_rate_pct'] = (acc_summary['win_rate']*100).round(1)

print("=== All 32 accounts by total PnL ===")
print(acc_summary[['total_pnl','avg_pnl','win_rate_pct','trade_count','coins_traded']].to_string())

profitable_pct = (acc_summary['total_pnl'] > 0).mean()
top10_pnl = acc_summary['total_pnl'].head(3).sum() / acc_summary[acc_summary['total_pnl']>0]['total_pnl'].sum()
print(f"\nProfitable accounts: {(acc_summary['total_pnl']>0).sum()} / {len(acc_summary)} ({profitable_pct:.1%})")
print(f"Top 3 accounts hold {top10_pnl:.1%} of total profits")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
rank = range(1, len(acc_summary)+1)
colors_acc = ['#D85A30' if v < 0 else '#1D9E75' for v in acc_summary['total_pnl']]
axes[0].scatter(rank, acc_summary['total_pnl'], c=colors_acc, s=60, alpha=0.8, zorder=3)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_title('Account PnL by rank', fontweight='bold')
axes[0].set_xlabel('Rank'); axes[0].set_ylabel('Total PnL (USD)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M' if abs(x)>=1e6 else f'${x/1e3:.0f}K'))

axes[1].bar(range(len(acc_summary)), acc_summary['total_pnl'], color=colors_acc, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Total PnL per account', fontweight='bold')
axes[1].set_xlabel('Account index'); axes[1].set_ylabel('Total PnL (USD)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M' if abs(x)>=1e6 else f'${x/1e3:.0f}K'))

plt.tight_layout(); plt.savefig('account_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Statistical Validation

In [ ]:
# ── 8. Statistical tests ─────────────────────────────────────────────────────
groups = [merged[merged['classification']==s]['Closed PnL'].dropna() for s in SENT_ORDER if len(merged[merged['classification']==s]) > 0]
group_names = [s for s in SENT_ORDER if len(merged[merged['classification']==s]) > 0]

print("=== Kruskal-Wallis: Does sentiment significantly affect PnL? ===")
stat, pval = kruskal(*groups)
print(f"H = {stat:.2f},  p = {pval:.4e}")
print("→ Significant difference across sentiment regimes" if pval < 0.05 else "→ Not significant")

print("\n=== Mann-Whitney: Extreme Greed vs Extreme Fear ===")
eg = merged[merged['classification']=='Extreme Greed']['Closed PnL'].dropna()
ef = merged[merged['classification']=='Extreme Fear']['Closed PnL'].dropna()
stat2, pval2 = mannwhitneyu(eg, ef, alternative='greater')
print(f"U = {stat2:.0f},  p = {pval2:.4e}")
print("→ Extreme Greed PnL significantly higher" if pval2 < 0.05 else "→ Not significant")

print("\n=== Binomial test: Extreme Greed win rate vs 50% baseline ===")
eg_wins = merged[merged['classification']=='Extreme Greed']['win']
res = binomtest(int(eg_wins.sum()), len(eg_wins), p=0.5, alternative='greater')
print(f"Win rate: {eg_wins.mean():.3f}   p = {res.pvalue:.4e}")
print("→ Win rate significantly above 50%" if res.pvalue < 0.05 else "→ Not significant")

print("\n=== Pearson: FG index (monthly) vs avg PnL ===")
r, p = stats.pearsonr(monthly['value'], monthly['avg_pnl'])
print(f"r = {r:.3f},  p = {p:.3f}")


## 9. Key Findings & Recommendations

In [ ]:
# ── 9. Summary ───────────────────────────────────────────────────────────────
print("=" * 70)
print("KEY FINDINGS — Hyperliquid × Bitcoin Fear & Greed Index")
print("=" * 70)

findings = [
    ("Extreme Greed = best performance",
     "Extreme Greed periods produced the highest avg PnL ($130) and win rate (89%).",
     "Counter to 'buy the fear' theory — this cohort thrives in euphoric markets."),
    ("BUYs dominate in Fear; SELLs dominate in Extreme Greed",
     "In Fear, BUYs averaged $210 vs SELLs $69. In Extreme Greed, SELLs averaged $176 vs BUYs $29.",
     "Traders profit by buying dips and shorting tops — a textbook contrarian swing approach."),
    ("Fear drives oversized positions",
     "Avg trade size during Fear: $8,041 — nearly 3× the Extreme Greed level ($2,780).",
     "Position sizing is sentiment-driven; traders deploy more capital when fearful."),
    ("Aug 2024 — only month with deeply negative PnL (-$310)",
     "FG index dropped to 35 (Fear). High position sizes + sudden drawdown = blowout.",
     "Risk management must adjust with sentiment; large Fear bets can backfire badly."),
    ("SOL & ETH deliver best alpha per trade",
     "SOL avg $326/trade, ETH $252/trade — far above BTC ($79) and HYPE ($61).",
     "FARTCOIN is the only top-10 coin with negative avg PnL (-$47). Meme coins are traps."),
    ("90.6% of accounts are profitable",
     "29 of 32 accounts ended positive. Top trader: +$2.14M. Worst: -$168K.",
     "This is an elite trader cohort — not a representative retail sample."),
]

for title, finding, implication in findings:
    print(f"\n▸ {title}")
    print(f"  Finding:     {finding}")
    print(f"  Implication: {implication}")

print("\n" + "=" * 70)
print("RECOMMENDED TRADING STRATEGY BASED ON FINDINGS:")
print("-" * 70)
print("1. Monitor FG index daily. Trade BUY side in Fear (FG < 45),")
print("   SELL/SHORT side in Extreme Greed (FG > 75).")
print("2. Largest position allocations in Fear zones — this cohort does this")
print("   successfully. But size down during extreme Fear (<25) — Aug 2024 shows risk.")
print("3. Prioritise SOL, ETH over BTC for better avg PnL per trade.")
print("4. Avoid meme coins (FARTCOIN = only negative-PnL top coin).")
print("5. Extreme Greed (FG > 75) is not a time to sit out — it's the single")
print("   best-performing sentiment regime for this Hyperliquid cohort.")
